In [3]:
# Cell 1 - imports & helpers
import json, pickle, os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from math import radians, sin, cos, asin, sqrt

# Haversine distance
def haversine_km(a, b):
    lat1, lon1 = a; lat2, lon2 = b
    lat1, lon1, lat2, lon2 = map(radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1; dlon = lon2 - lon1
    A = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return 6371 * 2 * asin(min(1, np.sqrt(A)))

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [4]:
# Cell 2 - load saved GNN model and artifacts (adjust paths to your saved files)
ARTIFACT_PKL = "model_artifacts.pkl"   # from your earlier code
GNN_MODEL_PTH = "gnn_model.pt"
GRAPH_DATA_PTH = "graph_data.pt"       # optional: saved Data object (if you saved)

assert os.path.exists(ARTIFACT_PKL), "model_artifacts.pkl not found. Save it first."

with open(ARTIFACT_PKL, 'rb') as f:
    artifacts = pickle.load(f)

node_index = artifacts['node_index']          # key -> idx
mlb = artifacts['mlb']                        # MultiLabelBinarizer used earlier
vect = artifacts['vect']                      # TF-IDF vectorizer used earlier
scaler = artifacts['scaler']                  # numeric scaler used earlier
safari_nodes = artifacts['safari_nodes']      # list of safari dicts (enriched JSON entries)

# Load your GNN model architecture (must match saved model)
# Recreate model class or import the same class as training script
class GCNLinkPredictor(nn.Module):
    def __init__(self, in_channels, hidden_channels=64):
        super().__init__()
        from torch_geometric.nn import GCNConv
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.conv2(x, edge_index)
        return x

# We need data.x and data.edge_index to forward the model for embeddings.
# If you saved the graph Data object earlier as 'graph_data.pt', load it; otherwise reconstruct using build_graph_from_json
if os.path.exists("graph_data.pt"):
    graph_data = torch.load("graph_data.pt")
else:
    # If not saved, you must re-run your build_graph_from_json to produce Data object.
    raise FileNotFoundError("graph_data.pt not found — generate the Data object via build_graph_from_json and save it.")

in_channels = graph_data.num_features
model = GCNLinkPredictor(in_channels, hidden_channels=64)
model.load_state_dict(torch.load(GNN_MODEL_PTH))
model.eval()

# Compute GNN embeddings for all nodes
with torch.no_grad():
    node_embeddings = model(graph_data.x, graph_data.edge_index)  # torch tensor (N, D)
node_embeddings = node_embeddings.cpu().numpy()
# For convenience, build a mapping from safari index (0..len(safari_nodes)-1) -> embedding
# Ensure the node_index mapping uses keys like "SAF:..."
safari_keys = [k for k in node_index.keys() if k.startswith("SAF:")]
safari_indices = [node_index[k] for k in safari_keys]
safari_embeddings = node_embeddings[safari_indices, :]  # shape (num_safaris, D)
# We assume safari_nodes list indices align with safari_keys order; if not, create an index map to align.
print("Loaded GNN embeddings:", safari_embeddings.shape)


FileNotFoundError: graph_data.pt not found — generate the Data object via build_graph_from_json and save it.